# ASTERIA EKF parameter search

This notebook can be used to do a bayesian search for the optimal EKF parameters.

Note: if use_storage is enabled, a sqliteDb with the name `sqlite:///ekf_optimization.db` is created. Use the following command in the db directory to open optuna dashboard: `optuna-dashboard sqlite:///ekf_optimization.db`

In [ ]:
from pathlib import Path

import numpy as np

from core import SimulationResult
from evaluation.aligners import NearestTimeIndexAligner
from evaluation.metrics import PositionRMSE, VelocityRMSE, AttitudeRMSE
from evaluation.runners import RustRunnerAsteria
from evaluation.search import BayesianSearch, SearchParams, FloatParam

# fetch every sim in rocketpy folder
EXCLUDE_LIST = {"asteria_evaluation"}
simulations_dir = Path("../../simulated_data/rocketpy/")
simulations: list[SimulationResult] = [
    SimulationResult.load(str(folder / f"{folder.name}.pkl"))
    for folder in sorted(simulations_dir.iterdir())
    if folder.is_dir()
       and (folder / f"{folder.name}.pkl").exists()
       and folder.name not in EXCLUDE_LIST
]


In [ ]:
runner_factory = lambda p: RustRunnerAsteria(accel_stddev=p["accel_stddev"],
                                             gyro_stddev=p["gyro_stddev"],
                                             gnss_stddev=np.array([p["gnss_xy"], p["gnss_xy"], p["gnss_z"]]))
search_params = SearchParams(
    accel_stddev=FloatParam(0.01, 0.3, log=True),
    gyro_stddev=FloatParam(0.0005, 0.03, log=True),
    gnss_xy=FloatParam(0.5, 10.0),
    gnss_z=FloatParam(1.0, 20.0),
)
metrics = [PositionRMSE()]
objective = lambda m: m["position_rmse"].value
aligner = NearestTimeIndexAligner()
n_trials = 50
directions = "minimize"
study_name = "ASTERIA param search"

search_minimal = BayesianSearch(runner_factory, search_params, metrics, objective, aligner, n_trials, directions,
                                study_name, use_storage=True, )
result = search_minimal.run(simulations)

In [ ]:
result.to_dataframe()

In [ ]:
result.aggregate()

In [ ]:
result.best_params("position_rmse")

In [ ]:
# Code for multi-objective search:

# search = BayesianSearch(
#     runner_factory=lambda p: RustRunnerAsteria(accel_stddev=p["accel_stddev"],
#                                                gyro_stddev=p["gyro_stddev"],
#                                                gnss_stddev=np.array([p["gnss_xy"], p["gnss_xy"], p["gnss_z"]]),
#                                                use_noisy=True,
#                                                do_correction=True, ),
#     search_params=SearchParams(
#         accel_stddev=FloatParam(0.01, 1.0, log=True),
#         gyro_stddev=FloatParam(0.001, 0.1, log=True),
#         gnss_xy=FloatParam(0.5, 10.0),
#         gnss_z=FloatParam(1.0, 20.0),
#     ),
#     metrics=[PositionRMSE(), VelocityRMSE(), AttitudeRMSE()],
#     objective=lambda m: [m["position_rmse"].value, m["velocity_rmse"].value, m["attitude_rmse"].value],
#     aligner=NearestTimeIndexAligner(),
#     directions=["minimize", "minimize", "minimize"],
#     n_trials=3,
#     study_name="ASTERIA param search",
#     use_storage=True,
# )
# result = search.run(simulations)

In [ ]:
# result.pareto_dataframe()